# F02 · Walk-Forward Return Prediction

**Paper:** Kozak, S., Nagel, S., & Santosh, S. (2020). Shrinking the cross-section. *Journal of Financial Economics, 135*(1), 271-292.

This notebook demonstrates walk-forward out-of-sample return prediction using `ReturnPredictor` with expanding and rolling windows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from empirlab.finance import ReturnPredictor, MLFactorModel
from empirlab.utils.metrics import ic, rmse
np.random.seed(42)

## 1 · Generate Synthetic Panel Data

In [ ]:
n_periods, n_stocks, n_chars = 60, 100, 20
rng = np.random.default_rng(0)
dates = pd.date_range('2018-01-01', periods=n_periods, freq='ME')
# True signals: first 3 characteristics predict returns
beta_true = np.zeros(n_chars); beta_true[:3] = [0.5, 0.3, -0.2]
chars_all, rets_all = [], []
for t in range(n_periods):
    X_t = rng.standard_normal((n_stocks, n_chars))
    r_t = X_t @ beta_true + rng.standard_normal(n_stocks) * 0.05
    chars_all.append(X_t); rets_all.append(r_t)
chars_3d = np.stack(chars_all)   # (T, N, K)
rets_2d  = np.stack(rets_all)    # (T, N)
print(f"Chars: {chars_3d.shape}, Returns: {rets_2d.shape}")

## 2 · Walk-Forward OOS Prediction

In [ ]:
rp = ReturnPredictor(model='enet', window='expanding', min_train=24)
oos_ics, oos_r2s = [], []
train_size = 24
for t in range(train_size, n_periods - 1):
    X_train = chars_3d[:t].reshape(-1, n_chars)
    y_train = rets_2d[:t].ravel()
    X_test  = chars_3d[t]
    y_test  = rets_2d[t + 1]
    rp.fit(X_train, y_train)
    y_hat = rp.predict(X_test)
    oos_ics.append(ic(y_hat, y_test))
    oos_r2s.append(rp.r2_oos(X_test, y_test))
print(f"Mean OOS IC:    {np.mean(oos_ics):.4f}")
print(f"Mean OOS R^2:   {np.mean(oos_r2s):.4f}")

## 3 · Plot OOS IC over Time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dates[train_size+1:], oos_ics, alpha=0.7, label='Monthly OOS IC')
ax.axhline(np.mean(oos_ics), color='red', ls='--', label=f'Mean IC={np.mean(oos_ics):.3f}')
ax.axhline(0, color='gray', ls=':')
ax.set_title('Walk-Forward OOS Information Coefficient'); ax.legend()
plt.tight_layout(); plt.show()

## 4 · Compare Methods

In [ ]:
methods = ['lasso', 'ridge', 'enet', 'forest']
results = {}
for method in methods:
    _ics = []
    m = ReturnPredictor(model=method, window='expanding', min_train=24)
    for t in range(train_size, n_periods - 1):
        X_tr = chars_3d[:t].reshape(-1, n_chars)
        y_tr = rets_2d[:t].ravel()
        m.fit(X_tr, y_tr)
        _ics.append(ic(m.predict(chars_3d[t]), rets_2d[t+1]))
    results[method] = np.mean(_ics)
print(pd.Series(results, name='Mean OOS IC').sort_values(ascending=False).to_string())